# Demo 3 — MLflow Prompt Tracker
## Session 5: Core Prompt Engineering Techniques

**Goal:** Demonstrate that prompts should be versioned and measured like code.

**Task:** Bug severity classification (P0-P3 only) evaluated on a 10-item golden test set.

**What we're comparing:**
- Prompt v1: minimal zero-shot — no role, no format spec
- Prompt v2: full PCT + context + JSON schema — production-ready

**What to watch:** The full prompt text is displayed before evaluation so you can see exactly what changed between versions.

**Expected outcome:** v2 should achieve higher severity accuracy than v1. Both versions are logged to MLflow so you can compare them in the UI side by side.

> **Connection to Session 4:** You used MLflow to track model experiments. This is the same workflow applied to prompt versions.

---
**Pre-session setup:** `mlflow ui --port 5000` in a separate terminal, then open http://localhost:5000

In [1]:
# -- Setup -----------------------------------------------------------------
import os, json, re, textwrap
import mlflow
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display

load_dotenv()

client = OpenAI(base_url=os.environ["API_BASE_URL"], api_key=os.environ["API_KEY"])
MODEL = os.environ.get("MODEL_NAME", "claude-sonnet-4-6")

mlflow.set_experiment("session5-prompt-versioning")

def show_prompt(system, user_template, label):
    w = 68
    print(f" {label} ".center(w, "="))
    print("[ SYSTEM ]")
    for line in system.strip().split("\n"):
        print(f"  {line}")
    print()
    print("[ USER TEMPLATE ]")
    for line in user_template.strip().split("\n"):
        print(f"  {line}")
    print("=" * w)
    print()

print(f"Model: {MODEL}")
print(f"MLflow URI: {mlflow.get_tracking_uri()}")

Model: llama3.2:1b
MLflow URI: sqlite:////Users/girishp/Code/AI-LLM%20Fundamentals%20Workshop%20Series/Session%2005%20-%20Core%20Prompt%20Engineering%20Techniques/05-demo/notebooks/mlflow.db


## Golden Test Set
10 labeled bug reports. Each has a known correct severity (P0-P3).
**Evaluation metric:** Severity accuracy only — did the model get P0/P1/P2/P3 right?

> **Why severity only?** Team assignment requires exact string matching which is unreliable for a 1B model. Severity is the most operationally important signal and is easier to extract reliably.

In [2]:
TEST_SET = [
    {"bug": "Login returns 500 for all users. DB connection pool exhausted.",                              "severity": "P0"},
    {"bug": "Payment confirmation emails not sent. 40% of transactions. Stripe webhooks show succeeded.", "severity": "P0"},
    {"bug": "Password reset emails arrive 45 minutes late. SendGrid shows queued.",                       "severity": "P1"},
    {"bug": "Product images not loading on iOS 17 Safari only. CSS transform issue.",                     "severity": "P2"},
    {"bug": "Checkout button label says 'Submit Order' instead of 'Place Order'.",                        "severity": "P3"},
    {"bug": "Search results return duplicates for 10% of queries. Elasticsearch index sync issue.",       "severity": "P2"},
    {"bug": "Refund processing stopped. All refund API calls return 422. 100% of refund requests.",       "severity": "P0"},
    {"bug": "Dashboard shows revenue totals off by ~2% due to timezone calculation bug.",                 "severity": "P2"},
    {"bug": "New user welcome email has a broken image link. Affects new signups only.",                  "severity": "P3"},
    {"bug": "Inventory goes negative during flash sales due to race condition. Orders oversold.",          "severity": "P0"},
]

def extract_severity(text):
    m = re.search(r"\b(P[0-3])\b", text, re.IGNORECASE)
    return m.group(1).upper() if m else None

df_test = pd.DataFrame(TEST_SET)
print(f"Test set: {len(TEST_SET)} items")
print(f"Severity distribution: {dict(pd.Series([t['severity'] for t in TEST_SET]).value_counts())}")
display(df_test)

Test set: 10 items
Severity distribution: {'P0': np.int64(4), 'P2': np.int64(3), 'P3': np.int64(2), 'P1': np.int64(1)}


,bug,severity
0,Login returns 500 for all users. DB connection...,P0
1,Payment confirmation emails not sent. 40% of t...,P0
2,Password reset emails arrive 45 minutes late. ...,P1
3,Product images not loading on iOS 17 Safari on...,P2
4,Checkout button label says 'Submit Order' inst...,P3
5,Search results return duplicates for 10% of qu...,P2
6,Refund processing stopped. All refund API call...,P0
7,Dashboard shows revenue totals off by ~2% due ...,P2
8,New user welcome email has a broken image link...,P3
9,Inventory goes negative during flash sales due...,P0


## Prompt v1 — Minimal Zero-Shot

In [3]:
V1_SYSTEM = "You are a helpful engineering assistant."
V1_USER_TEMPLATE = "What is the severity (P0, P1, P2, or P3) of this bug?\n\n{bug}\n\nSeverity:"

show_prompt(V1_SYSTEM, V1_USER_TEMPLATE, "Prompt v1.0.0 - Minimal Zero-Shot")

================ Prompt v1.0.0 - Minimal Zero-Shot =================
[ SYSTEM ]
  You are a helpful engineering assistant.

[ USER TEMPLATE ]
  What is the severity (P0, P1, P2, or P3) of this bug?
  
  {bug}
  
  Severity:



## Prompt v2 — PCT + Context + Format

In [4]:
V2_SYSTEM = """You are a senior software engineer triaging production bugs.

Severity model (use exactly one):
- P0: Revenue impact or data loss, more than 20% of users affected
- P1: Major feature broken, 5-20% of users affected, no workaround
- P2: Feature degraded, fewer than 5% affected, or workaround exists
- P3: Cosmetic issue, low impact

Respond with ONLY the severity label: P0, P1, P2, or P3. Nothing else."""

V2_USER_TEMPLATE = "Bug report:\n{bug}\n\nSeverity:"

show_prompt(V2_SYSTEM, V2_USER_TEMPLATE, "Prompt v2.0.0 - PCT + Severity Model")

=============== Prompt v2.0.0 - PCT + Severity Model ===============
[ SYSTEM ]
  You are a senior software engineer triaging production bugs.
  
  Severity model (use exactly one):
  - P0: Revenue impact or data loss, more than 20% of users affected
  - P1: Major feature broken, 5-20% of users affected, no workaround
  - P2: Feature degraded, fewer than 5% affected, or workaround exists
  - P3: Cosmetic issue, low impact
  
  Respond with ONLY the severity label: P0, P1, P2, or P3. Nothing else.

[ USER TEMPLATE ]
  Bug report:
  {bug}
  
  Severity:



## Evaluate Both Versions on the Golden Test Set

In [5]:
def run_prompt(system, user_template, label):
    results = []
    for item in TEST_SET:
        user = user_template.format(bug=item["bug"])
        resp = client.chat.completions.create(
            model=MODEL, max_tokens=10,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": user}]
        )
        predicted = extract_severity(resp.choices[0].message.content)
        correct = predicted == item["severity"]
        results.append({"expected": item["severity"], "predicted": predicted or "?", "correct": correct})
        status = "OK  " if correct else "FAIL"
        print(f"  [{status}] expected={item['severity']}  got={predicted or '?'}")
    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\n{label} accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")
    return results, accuracy

print("Evaluating Prompt v1...")
v1_results, v1_accuracy = run_prompt(V1_SYSTEM, V1_USER_TEMPLATE, "v1")

print()
print("Evaluating Prompt v2...")
v2_results, v2_accuracy = run_prompt(V2_SYSTEM, V2_USER_TEMPLATE, "v2")

print()
print(f"Accuracy delta: v1={v1_accuracy:.0%}  v2={v2_accuracy:.0%}  delta={v2_accuracy - v1_accuracy:+.0%}")

Evaluating Prompt v1...


  [FAIL] expected=P0  got=?


  [FAIL] expected=P0  got=?


  [FAIL] expected=P1  got=P3


  [FAIL] expected=P2  got=?


...

  [FAIL] expected=P0  got=P1
  [OK  ] expected=P0  got=P0


  [OK  ] expected=P1  got=P1
  [FAIL] expected=P2  got=P0


  [FAIL] expected=P3  got=P2
  [FAIL] expected=P2  got=P0


  [OK  ] expected=P0  got=P0
  [OK  ] expected=P2  got=P2


  [OK  ] expected=P3  got=P3
  [FAIL] expected=P0  got=P1

v2 accuracy: 50% (5/10)

Accuracy delta: v1=0%  v2=50%  delta=+50%


## Log Both Versions to MLflow

In [6]:
with mlflow.start_run(run_name="prompt-v1"):
    mlflow.log_param("version",      "v1.0.0")
    mlflow.log_param("description",  "Minimal zero-shot")
    mlflow.log_param("model",        MODEL)
    mlflow.log_param("system_prompt", V1_SYSTEM)
    mlflow.log_param("user_template", V1_USER_TEMPLATE)
    mlflow.log_metric("severity_accuracy", v1_accuracy)
    mlflow.log_metric("correct_count",     sum(r["correct"] for r in v1_results))
print(f"Logged v1: accuracy={v1_accuracy:.0%}")

with mlflow.start_run(run_name="prompt-v2"):
    mlflow.log_param("version",      "v2.0.0")
    mlflow.log_param("description",  "PCT + severity model + format spec")
    mlflow.log_param("model",        MODEL)
    mlflow.log_param("system_prompt", V2_SYSTEM)
    mlflow.log_param("user_template", V2_USER_TEMPLATE)
    mlflow.log_metric("severity_accuracy", v2_accuracy)
    mlflow.log_metric("correct_count",     sum(r["correct"] for r in v2_results))
print(f"Logged v2: accuracy={v2_accuracy:.0%}")

print()
print("Next: open http://localhost:5000")
print("  -> select both runs -> Compare")
print("  -> you will see severity_accuracy side by side + full prompt text diff")

Logged v1: accuracy=0%
Logged v2: accuracy=50%

Next: open http://localhost:5000
  -> select both runs -> Compare
  -> you will see severity_accuracy side by side + full prompt text diff


## Summary

In [7]:
comparison = pd.DataFrame([
    {"Version": "v1.0.0", "Description": "Minimal zero-shot",
     "Severity Accuracy": f"{v1_accuracy:.0%}", "Correct / 10": sum(r['correct'] for r in v1_results)},
    {"Version": "v2.0.0", "Description": "PCT + severity model + format",
     "Severity Accuracy": f"{v2_accuracy:.0%}", "Correct / 10": sum(r['correct'] for r in v2_results)},
])
display(comparison)
print()
print("Key takeaway:")
print("  The prompt diff is a 30-second read. The accuracy delta tells you it is worth shipping.")
print("  Without MLflow: which version is in production right now? Can you answer that?")

,Version,Description,Severity Accuracy,Correct / 10
0,v1.0.0,Minimal zero-shot,0%,0
1,v2.0.0,PCT + severity model + format,50%,5



Key takeaway:
  The prompt diff is a 30-second read. The accuracy delta tells you it is worth shipping.
  Without MLflow: which version is in production right now? Can you answer that?
